# Skidorternas vintrar – Sälen, Åre, Tärnaby

Fristående notebook. Hela `skiclimate`-paketet ligger i cellerna nedan och skrivs till disk när du kör dem, så du behöver inte klona något. Fungerar i **Google Colab** och **Microsoft Fabric**.

**Kör allt uppifrån och ner.** Första riktiga nedladdningen tar 10–30 minuter. Rådatan sparas under `data/` så steg 2 och 3 kan köras om utan nätverk.

Sätt `SYNTHETIC = True` i cellen nedan om du bara vill se att allt fungerar utan att vänta på nätverket.

In [ ]:
SYNTHETIC = False        # True = påhittad data, inget nätverk, klart på 15 s
RESORTS = 'all'          # eller t.ex. 'are,salen'
ELEVATION = 'base'       # 'base' = byn, 'top' = högsta liftpunkten
ERA5_SNOW = False        # True = hämta ERA5-snödjup per timme (långsamt, ~85 anrop per ort)

%pip install -q pandas numpy scipy scikit-learn statsmodels matplotlib pyarrow requests

import os
os.makedirs('skiclimate', exist_ok=True)

## Koden

Varje cell skriver en modul. Rör dem inte om du inte vill; tröskelvärden och koordinater ligger i `config.py`.

In [ ]:
%%writefile skiclimate/__init__.py
"""skiclimate – väderhistorik och säsongsanalys för Sälen, Åre och Tärnaby.

Pipeline i tre steg:

    download  ->  SMHI-stationer + ERA5 (Open-Meteo) + CMIP6 (Open-Meteo climate)
    features  ->  en rad per ort och vinter (säsongslängd, tödagar, frostnätter ...)
    analyze   ->  trender, brytpunkter, känslighet mot global temperatur, ML-projektion

Allt körs från ``run_pipeline.py`` eller från en notebook (Colab / Fabric).
"""

__version__ = "0.1.0"


In [ ]:
%%writefile skiclimate/config.py
"""Alla siffror man kan bråka om, samlade på ett ställe.

Koordinater, stationssök, parameter-id:n hos SMHI, tröskelvärden för
säsongsmått och vilka klimatmodeller vi hämtar. Ändra här, inte i koden.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

# ---------------------------------------------------------------------------
# Orter
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Resort:
    key: str
    name: str
    lat: float
    lon: float
    #: byn / liftbotten, meter över havet
    base_elevation: int
    #: högsta liftburna punkt, meter över havet
    top_elevation: int
    #: hur långt bort vi letar SMHI-stationer (km)
    station_radius_km: float = 60.0
    #: stationsnamn (delsträng, skiftlägesokänslig) som ska prioriteras om de finns
    preferred_stations: tuple[str, ...] = ()


RESORTS: dict[str, Resort] = {
    "salen": Resort(
        key="salen",
        name="Sälen",
        lat=61.1600,
        lon=13.2620,
        base_elevation=420,
        top_elevation=890,
        preferred_stations=("Sälen", "Idre", "Malung", "Särna", "Storbron", "Höljes"),
    ),
    "are": Resort(
        key="are",
        name="Åre",
        lat=63.3990,
        lon=13.0810,
        base_elevation=380,
        top_elevation=1420,
        preferred_stations=("Åre", "Storlien", "Sylarna", "Duved", "Järpen", "Hallen", "Enafors"),
    ),
    "tarnaby": Resort(
        key="tarnaby",
        name="Tärnaby",
        lat=65.7180,
        lon=15.3120,
        base_elevation=470,
        top_elevation=1000,
        preferred_stations=("Tärnaby", "Hemavan", "Storuman", "Klimpfjäll", "Gielas", "Stekenjokk"),
    ),
}


# ---------------------------------------------------------------------------
# SMHI:s öppna data (meteorologiska observationer)
# ---------------------------------------------------------------------------

SMHI_BASE = "https://opendata-download-metobs.smhi.se/api/version/1.0"

#: parameter-id -> (kolumnnamn hos oss, enhet vi vill ha, skalfaktor från SMHI:s enhet)
#: Dygnsparametrar först; de är vad säsongsanalysen faktiskt behöver.
SMHI_DAILY_PARAMETERS: dict[int, tuple[str, str, float]] = {
    2: ("t_mean", "°C", 1.0),      # Lufttemperatur, medel 1 dygn
    19: ("t_min", "°C", 1.0),      # Lufttemperatur, min per dygn
    20: ("t_max", "°C", 1.0),      # Lufttemperatur, max per dygn
    5: ("precip", "mm", 1.0),      # Nederbördsmängd, summa 1 dygn (kl 06)
    8: ("snow_depth", "cm", 100.0),  # Snödjup kl 06 – SMHI ger meter, vi vill ha cm
}

#: Timparametrar. Stora filer, behövs bara om man vill räkna våt temperatur
#: för snökanoner på riktigt. Avstängda som standard.
SMHI_HOURLY_PARAMETERS: dict[int, tuple[str, str, float]] = {
    1: ("t_hour", "°C", 1.0),      # Lufttemperatur, momentan, varje timme
    6: ("rh_hour", "%", 1.0),      # Relativ luftfuktighet
    4: ("wind_hour", "m/s", 1.0),  # Vindhastighet, 10-min medel
}

#: SMHI:s kvalitetskoder. G = granskad, Y = ogranskad/misstänkt, R = ...tveksam.
#: Vi behåller G och Y, kastar allt annat. Y är oftast bara "inte hunnit granska".
SMHI_KEEP_QUALITY = ("G", "Y")


# ---------------------------------------------------------------------------
# Open-Meteo: ERA5-reanalys (1940-) och CMIP6-projektioner (1950-2050)
# ---------------------------------------------------------------------------

OPENMETEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
OPENMETEO_CLIMATE = "https://climate-api.open-meteo.com/v1/climate"

ERA5_START = "1940-01-01"

ERA5_DAILY_VARIABLES = (
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "wind_speed_10m_max",
    "shortwave_radiation_sum",
)

#: snödjup finns bara per timme i arkivet. Hämtas år för år och pressas till dygn.
ERA5_HOURLY_SNOW_VARIABLE = "snow_depth"

#: Samma sju HighResMIP-modeller som webbsidans "warming"-vy. Enda scenariot
#: Open-Meteo tillhandahåller är SSP5-8.5, vilket är det pessimistiska. Säg det
#: högt varje gång siffrorna presenteras.
CMIP6_MODELS = (
    "EC_Earth3P_HR",
    "MRI_AGCM3_2_S",
    "HiRAM_SIT_HR",
    "CMCC_CM2_VHR4",
    "FGOALS_f3_H",
    "NICAM16_8S",
    "MPI_ESM1_2_XR",
)
CMIP6_START = "1950-01-01"
CMIP6_END = "2050-12-31"
CMIP6_DAILY_VARIABLES = (
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "snowfall_sum",
)

#: NASA GISTEMP, global medeltemperaturanomali. Används för att koppla lokal
#: vintertemperatur till global uppvärmning. Frivillig – pipelinen klarar sig utan.
GISTEMP_URL = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"


# ---------------------------------------------------------------------------
# Säsongsdefinitioner
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class SeasonRules:
    #: en vinter börjar i juli så att ingen vinter delas på två kalenderår
    year_start_month: int = 7
    #: "kärnvinter" – när en tödag faktiskt gör skada
    core_winter: tuple[int, int] = (12, 3)
    #: perioden snökanonerna får köra
    making_season: tuple[int, int] = (11, 3)
    #: dygnsmin under detta ≈ våt temperatur under -2 °C i fjällen (grov proxy)
    snowmaking_tmin: float = -4.0
    #: dygnsmax över detta mitt i vintern räknas som tö
    thaw_above: float = 2.0
    #: klassiska 100-dagarsregeln: minst så här djupt, minst så här många dagar
    reliable_depth_cm: float = 30.0
    reliable_days: int = 100
    #: snötäcke över huvud taget
    cover_depth_cm: float = 1.0
    #: så många dygn i rad under noll öppnar/stänger "termisk vinter"
    run_length: int = 5
    #: graddagsmodell: smältning per plusgrad och dygn (mm vattenekvivalent)
    melt_factor: float = 3.5
    #: densitet i satt snötäcke (kg/m³), för att göra mm vatten till cm snö
    pack_density: float = 300.0
    #: nederbörd faller som snö när dygnsmedlet är under detta
    snow_temp_threshold: float = 1.0
    #: temperaturavtagande med höjd, °C per 100 m
    lapse_rate: float = 0.65


SEASON = SeasonRules()

#: WMO-normalperioderna vi jämför
PERIODS = {
    "past": (1961, 1990),
    "present": (1991, 2020),
    "future": (2031, 2050),
}


# ---------------------------------------------------------------------------
# Kataloger
# ---------------------------------------------------------------------------


@dataclass
class Paths:
    root: Path = field(default_factory=lambda: Path("data"))

    @property
    def raw(self) -> Path:
        return self.root / "raw"

    @property
    def processed(self) -> Path:
        return self.root / "processed"

    @property
    def output(self) -> Path:
        return self.root / "output"

    def ensure(self) -> "Paths":
        for p in (self.raw, self.processed, self.output):
            p.mkdir(parents=True, exist_ok=True)
        return self


# ---------------------------------------------------------------------------
# Färger för diagrammen. Tre orter = tre första platserna i en CVD-validerad
# palett (blå, orange, aqua). Lägg inte till en fjärde ort utan att validera.
# ---------------------------------------------------------------------------
RESORT_COLORS = {
    "salen": "#2a78d6",
    "are": "#eb6834",
    "tarnaby": "#1baf7a",
}


In [ ]:
%%writefile skiclimate/smhi.py
"""SMHI:s öppna observationsdata (metobs).

Två saker händer här:

1. Hitta stationer nära en ort som mäter en given parameter.
2. Ladda ner hela det korrigerade arkivet (CSV) plus de senaste månaderna,
   och göra om SMHI:s hemsnickrade CSV-dialekt till en vanlig DataFrame.

SMHI:s CSV är inte en CSV. Den är tre metadatablock, en tom rad, och sedan
en tabell vars kolumner heter olika saker beroende på parameter. Parsern nedan
letar reda på tabellen istället för att gissa var den börjar.
"""

from __future__ import annotations

import io
import logging
import math
import time
from dataclasses import dataclass

import pandas as pd
import requests

from .config import (
    RESORTS,
    SMHI_BASE,
    SMHI_DAILY_PARAMETERS,
    SMHI_HOURLY_PARAMETERS,
    SMHI_KEEP_QUALITY,
    Resort,
)

log = logging.getLogger(__name__)

_SESSION = requests.Session()
_SESSION.headers.update({"User-Agent": "skiclimate/0.1 (github.com/antonalin/AreWeather)"})


def _get(url: str, retries: int = 4, timeout: int = 120) -> requests.Response:
    """GET med backoff. SMHI:s servrar har dåliga dagar precis som alla andra."""
    delay = 2.0
    last: Exception | None = None
    for attempt in range(retries):
        try:
            r = _SESSION.get(url, timeout=timeout)
            if r.status_code == 404:
                r.raise_for_status()
            if r.status_code >= 500:
                raise requests.HTTPError(f"{r.status_code} från SMHI", response=r)
            r.raise_for_status()
            return r
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                raise
            last = e
        except requests.RequestException as e:
            last = e
        log.warning("SMHI %s misslyckades (försök %d/%d): %s", url, attempt + 1, retries, last)
        time.sleep(delay)
        delay *= 2
    raise RuntimeError(f"Gav upp på {url}: {last}")


# ---------------------------------------------------------------------------
# Stationer
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class Station:
    key: int
    name: str
    lat: float
    lon: float
    height: float
    active: bool
    from_year: int
    to_year: int
    distance_km: float


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Storcirkelavstånd. Jorden är rund, oavsett vad kommentarsfältet säger."""
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def list_stations(parameter: int) -> list[dict]:
    """Alla stationer som någonsin mätt parametern, rakt från SMHI."""
    r = _get(f"{SMHI_BASE}/parameter/{parameter}.json")
    return r.json().get("station", [])


def stations_near(resort: Resort, parameter: int, min_years: int = 10) -> list[Station]:
    """Stationer inom radien, sorterade på (föredragen först, sedan avstånd).

    ``min_years`` kastar stationer som bara stod uppe ett par säsonger; de
    tillför inget till en trendanalys men kostar en nedladdning var.
    """
    found: list[Station] = []
    for s in list_stations(parameter):
        d = haversine_km(resort.lat, resort.lon, s["latitude"], s["longitude"])
        if d > resort.station_radius_km:
            continue
        # SMHI ger epoch-millisekunder; ett år är ungefär så här många av dem
        from_year = pd.to_datetime(s["from"], unit="ms").year
        to_year = pd.to_datetime(s["to"], unit="ms").year
        if to_year - from_year < min_years:
            continue
        found.append(
            Station(
                key=int(s["key"]),
                name=s["name"],
                lat=s["latitude"],
                lon=s["longitude"],
                height=float(s.get("height", float("nan"))),
                active=bool(s.get("active", False)),
                from_year=from_year,
                to_year=to_year,
                distance_km=round(d, 1),
            )
        )

    prefs = tuple(p.lower() for p in resort.preferred_stations)

    def rank(st: Station) -> tuple[int, float]:
        hit = any(p in st.name.lower() for p in prefs)
        return (0 if hit else 1, st.distance_km)

    return sorted(found, key=rank)


# ---------------------------------------------------------------------------
# CSV-parsern
# ---------------------------------------------------------------------------

_DATE_COLUMNS = ("Representativt dygn", "Datum")
_META_COLUMNS = {"Kvalitet", "Tidsutsnitt:", "", "Tid (UTC)", "Från Datum Tid (UTC)", "Till Datum Tid (UTC)"}


def parse_smhi_csv(text: str, parameter: int) -> pd.DataFrame:
    """Gör en DataFrame ``[date, value, quality]`` av SMHI:s CSV-export.

    Fungerar för både dygns- och timparametrar. För timparametrar blir
    ``date`` en full tidsstämpel, för dygn en ren dag.
    """
    lines = text.splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.startswith("Datum;") or line.startswith("Från Datum"):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Hittade ingen datatabell i SMHI-svaret – formatet har ändrats eller filen är tom")

    table = "\n".join(lines[header_idx:])
    df = pd.read_csv(io.StringIO(table), sep=";", dtype=str, keep_default_na=False)
    # Sista kolumnerna är SMHI:s kvalitetsförklaring, en per rad, helt värdelös här
    df.columns = [c.strip() for c in df.columns]

    date_col = next((c for c in _DATE_COLUMNS if c in df.columns), None)
    if date_col is None:
        raise ValueError(f"Ingen datumkolumn bland {list(df.columns)}")

    value_col = next(
        (c for c in df.columns if c not in _META_COLUMNS and c not in _DATE_COLUMNS and not c.startswith("Unnamed")),
        None,
    )
    if value_col is None:
        raise ValueError(f"Ingen värdekolumn bland {list(df.columns)}")

    out = pd.DataFrame()
    if date_col == "Datum" and "Tid (UTC)" in df.columns:
        # Timdata: datum och tid i två kolumner, för att livet ska kännas längre
        out["date"] = pd.to_datetime(df["Datum"] + " " + df["Tid (UTC)"], errors="coerce", utc=True)
        if parameter in SMHI_DAILY_PARAMETERS:
            # dygnsparametrar med klockslag (snödjup kl 06, nederbörd kl 06): kasta klockan
            out["date"] = out["date"].dt.tz_localize(None).dt.normalize()
    else:
        out["date"] = pd.to_datetime(df[date_col], errors="coerce")

    out["value"] = pd.to_numeric(df[value_col].str.replace(",", "."), errors="coerce")
    out["quality"] = df["Kvalitet"].str.strip() if "Kvalitet" in df.columns else "G"
    out = out.dropna(subset=["date", "value"])
    out = out[out["quality"].isin(SMHI_KEEP_QUALITY)]
    return out.reset_index(drop=True)


def _fetch_period(parameter: int, station_key: int, period: str) -> pd.DataFrame | None:
    url = f"{SMHI_BASE}/parameter/{parameter}/station/{station_key}/period/{period}/data.csv"
    try:
        r = _get(url)
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            # Stationen har inte den perioden. Helt normalt, inget att gråta över.
            return None
        raise
    # SMHI säger utf-8. Ibland ljuger de. Å, ä och ö får stå sitt kast.
    try:
        text = r.content.decode("utf-8")
    except UnicodeDecodeError:
        text = r.content.decode("latin-1")
    return parse_smhi_csv(text, parameter)


def fetch_station_series(parameter: int, station_key: int) -> pd.DataFrame:
    """Korrigerat arkiv + senaste månaderna, ihopslagna utan dubbletter.

    Arkivet släpar ett par månader; ``latest-months`` täcker glappet. Där de
    överlappar vinner arkivet, eftersom det är granskat.
    """
    parts = []
    archive = _fetch_period(parameter, station_key, "corrected-archive")
    if archive is not None and not archive.empty:
        parts.append(archive)
    recent = _fetch_period(parameter, station_key, "latest-months")
    if recent is not None and not recent.empty:
        if parts:
            recent = recent[recent["date"] > parts[0]["date"].max()]
        parts.append(recent)
    if not parts:
        return pd.DataFrame(columns=["date", "value", "quality"])
    df = pd.concat(parts, ignore_index=True)
    return df.drop_duplicates("date", keep="first").sort_values("date").reset_index(drop=True)


# ---------------------------------------------------------------------------
# Hela nedladdningen för en ort
# ---------------------------------------------------------------------------


def download_resort(
    resort: Resort,
    parameters: dict[int, tuple[str, str, float]] | None = None,
    max_stations_per_parameter: int = 6,
    pause_s: float = 0.5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returnerar ``(observations, stations)`` i långt format.

    observations: resort, parameter, variable, station_key, date, value, quality
    stations:     resort, parameter, station_key, name, lat, lon, height, distance_km, ...
    """
    parameters = parameters or SMHI_DAILY_PARAMETERS
    obs_frames: list[pd.DataFrame] = []
    station_rows: list[dict] = []

    for pid, (variable, unit, scale) in parameters.items():
        try:
            candidates = stations_near(resort, pid)
        except Exception as e:  # noqa: BLE001 – vi vill logga och gå vidare, inte dö
            log.error("Kunde inte lista stationer för %s param %d: %s", resort.name, pid, e)
            continue
        log.info("%s: %d stationer för %s (param %d)", resort.name, len(candidates), variable, pid)

        for st in candidates[:max_stations_per_parameter]:
            try:
                series = fetch_station_series(pid, st.key)
            except Exception as e:  # noqa: BLE001
                log.error("%s/%s param %d: %s", resort.name, st.name, pid, e)
                continue
            time.sleep(pause_s)  # vi är gäster hos en myndighet, uppför oss
            if series.empty:
                continue
            series = series.assign(
                resort=resort.key,
                parameter=pid,
                variable=variable,
                unit=unit,
                station_key=st.key,
                value=series["value"] * scale,
            )
            obs_frames.append(series)
            station_rows.append(
                {
                    "resort": resort.key,
                    "parameter": pid,
                    "variable": variable,
                    "station_key": st.key,
                    "name": st.name,
                    "lat": st.lat,
                    "lon": st.lon,
                    "height": st.height,
                    "distance_km": st.distance_km,
                    "active": st.active,
                    "from_year": st.from_year,
                    "to_year": st.to_year,
                    "n_obs": len(series),
                    "first_obs": series["date"].min(),
                    "last_obs": series["date"].max(),
                }
            )
            log.info(
                "  %s (%d): %d värden %s..%s, %.0f m ö.h., %.0f km bort",
                st.name, st.key, len(series), series["date"].min().date(), series["date"].max().date(),
                st.height, st.distance_km,
            )

    obs = pd.concat(obs_frames, ignore_index=True) if obs_frames else pd.DataFrame(
        columns=["date", "value", "quality", "resort", "parameter", "variable", "unit", "station_key"]
    )
    return obs, pd.DataFrame(station_rows)


def download_all(resort_keys: list[str] | None = None, hourly: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    keys = resort_keys or list(RESORTS)
    params = dict(SMHI_DAILY_PARAMETERS)
    if hourly:
        params.update(SMHI_HOURLY_PARAMETERS)
    obs_all, st_all = [], []
    for k in keys:
        o, s = download_resort(RESORTS[k], params)
        obs_all.append(o)
        st_all.append(s)
    return pd.concat(obs_all, ignore_index=True), pd.concat(st_all, ignore_index=True)


In [ ]:
%%writefile skiclimate/openmeteo.py
"""Open-Meteo: ERA5-reanalys bakåt till 1940 och CMIP6-projektioner till 2050.

ERA5 är inte en mätning, det är en väderprognos som körts baklänges med alla
observationer inmatade. Den täcker varje dag sedan 1940 utan luckor, vilket
ingen svensk fjällstation gör. Priset är en gridruta på ~10-25 km som inte
skiljer på dalbotten och topp. Därför sparar vi rutans egen höjd och låter
feature-steget lapse-korrigera till den höjd vi faktiskt bryr oss om.

CMIP6-datat är sju HighResMIP-modeller, nedskalade och biaskorrigerade av
Open-Meteo. Bara SSP5-8.5 finns. Det är det varmaste scenariot – behandla
resultatet som en övre gräns, inte en prognos.
"""

from __future__ import annotations

import logging
import time
from datetime import date

import pandas as pd
import requests

from .config import (
    CMIP6_DAILY_VARIABLES,
    CMIP6_END,
    CMIP6_MODELS,
    CMIP6_START,
    ERA5_DAILY_VARIABLES,
    ERA5_HOURLY_SNOW_VARIABLE,
    ERA5_START,
    GISTEMP_URL,
    OPENMETEO_ARCHIVE,
    OPENMETEO_CLIMATE,
    Resort,
)

log = logging.getLogger(__name__)

_SESSION = requests.Session()
_SESSION.headers.update({"User-Agent": "skiclimate/0.1 (github.com/antonalin/AreWeather)"})


def _get_json(url: str, params: dict, retries: int = 5, timeout: int = 180) -> dict:
    """Open-Meteo svarar 429 när man är för ivrig. Då väntar vi. Länge."""
    delay = 3.0
    last: Exception | None = None
    for attempt in range(retries):
        try:
            r = _SESSION.get(url, params=params, timeout=timeout)
            if r.status_code == 429:
                raise requests.HTTPError("429 – för många anrop", response=r)
            r.raise_for_status()
            payload = r.json()
            if payload.get("error"):
                raise RuntimeError(payload.get("reason", "okänt fel från Open-Meteo"))
            return payload
        except (requests.RequestException, RuntimeError, ValueError) as e:
            last = e
            log.warning("Open-Meteo %s (försök %d/%d): %s", url, attempt + 1, retries, e)
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(f"Gav upp på {url} {params}: {last}")


def _daily_frame(payload: dict) -> pd.DataFrame:
    daily = payload["daily"]
    df = pd.DataFrame(daily)
    df["date"] = pd.to_datetime(df.pop("time"))
    return df


# ---------------------------------------------------------------------------
# ERA5
# ---------------------------------------------------------------------------


def _year_chunks(start: str, end: str, years_per_chunk: int) -> list[tuple[str, str]]:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    chunks = []
    cur = s
    while cur <= e:
        nxt = min(pd.Timestamp(year=cur.year + years_per_chunk, month=1, day=1) - pd.Timedelta(days=1), e)
        chunks.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt + pd.Timedelta(days=1)
    return chunks


def fetch_era5_daily(
    resort: Resort,
    start: str = ERA5_START,
    end: str | None = None,
    years_per_chunk: int = 15,
    pause_s: float = 1.0,
) -> tuple[pd.DataFrame, float]:
    """Dygnsvärden från ERA5 för ortens koordinat. Returnerar ``(df, grid_elevation)``.

    Hela 1940-idag i ett anrop går oftast bra, men Open-Meteo har ett
    timeout-humör, så vi delar upp i bitar och slipper börja om från noll.
    """
    end = end or (date.today() - pd.Timedelta(days=6)).strftime("%Y-%m-%d")  # ERA5 släpar ~5 dagar
    frames = []
    elevation = float("nan")
    for s, e in _year_chunks(start, end, years_per_chunk):
        payload = _get_json(
            OPENMETEO_ARCHIVE,
            {
                "latitude": resort.lat,
                "longitude": resort.lon,
                "start_date": s,
                "end_date": e,
                "daily": ",".join(ERA5_DAILY_VARIABLES),
                "timezone": "Europe/Stockholm",
                "models": "era5_land",  # 9 km istället för 25 – terrängen märks
            },
        )
        elevation = float(payload.get("elevation", elevation))
        frames.append(_daily_frame(payload))
        log.info("ERA5 %s %s..%s klart", resort.name, s, e)
        time.sleep(pause_s)
    df = pd.concat(frames, ignore_index=True).drop_duplicates("date").sort_values("date")
    df = df.rename(
        columns={
            "temperature_2m_mean": "t_mean",
            "temperature_2m_max": "t_max",
            "temperature_2m_min": "t_min",
            "precipitation_sum": "precip",
            "rain_sum": "rain",
            "snowfall_sum": "snowfall_cm",  # Open-Meteo ger cm snö, inte mm vatten
            "wind_speed_10m_max": "wind_max",
            "shortwave_radiation_sum": "radiation",
        }
    )
    df["resort"] = resort.key
    df["source"] = "era5_land"
    df["grid_elevation"] = elevation
    return df.reset_index(drop=True), elevation


def fetch_era5_snow_depth(
    resort: Resort,
    start: str = ERA5_START,
    end: str | None = None,
    pause_s: float = 1.5,
) -> pd.DataFrame:
    """Snödjup från ERA5-Land, timme för timme, år för år, ner till dygnsmedel/max.

    Ett år per anrop är 8 760 rader; 85 år är 85 anrop. Det tar en stund.
    Sätt på kaffet. Modellsnö är dessutom vattenekvivalent omräknad med en
    fast densitet, så absolutnivån är tveksam – trenden är det intressanta.
    """
    end = end or (date.today() - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    frames = []
    for s, e in _year_chunks(start, end, 1):
        payload = _get_json(
            OPENMETEO_ARCHIVE,
            {
                "latitude": resort.lat,
                "longitude": resort.lon,
                "start_date": s,
                "end_date": e,
                "hourly": ERA5_HOURLY_SNOW_VARIABLE,
                "timezone": "Europe/Stockholm",
                "models": "era5_land",
            },
        )
        h = pd.DataFrame(payload["hourly"])
        h["time"] = pd.to_datetime(h["time"])
        h["date"] = h["time"].dt.normalize()
        d = h.groupby("date")[ERA5_HOURLY_SNOW_VARIABLE].agg(["mean", "max"]).reset_index()
        d.columns = ["date", "snow_depth_era5_mean_cm", "snow_depth_era5_max_cm"]
        d[["snow_depth_era5_mean_cm", "snow_depth_era5_max_cm"]] *= 100.0  # meter -> cm
        frames.append(d)
        log.info("ERA5 snödjup %s %s klart", resort.name, s[:4])
        time.sleep(pause_s)
    df = pd.concat(frames, ignore_index=True)
    df["resort"] = resort.key
    return df


# ---------------------------------------------------------------------------
# CMIP6
# ---------------------------------------------------------------------------


def fetch_cmip6(resort: Resort, models: tuple[str, ...] = CMIP6_MODELS, pause_s: float = 2.0) -> tuple[pd.DataFrame, float]:
    """En modell per anrop så kolumnnamnen inte får modellnamnet fastklistrat."""
    frames = []
    elevation = float("nan")
    for m in models:
        payload = _get_json(
            OPENMETEO_CLIMATE,
            {
                "latitude": resort.lat,
                "longitude": resort.lon,
                "start_date": CMIP6_START,
                "end_date": CMIP6_END,
                "models": m,
                "daily": ",".join(CMIP6_DAILY_VARIABLES),
                "timezone": "Europe/Stockholm",
            },
        )
        elevation = float(payload.get("elevation", elevation))
        df = _daily_frame(payload)
        # När man bara begär en modell ska kolumnerna vara rena, men Open-Meteo
        # har bytt åsikt om det förr. Skala av suffixet om det dyker upp.
        df.columns = [c.replace(f"_{m}", "") for c in df.columns]
        df = df.rename(
            columns={
                "temperature_2m_mean": "t_mean",
                "temperature_2m_max": "t_max",
                "temperature_2m_min": "t_min",
                "precipitation_sum": "precip",
                "snowfall_sum": "snowfall_cm",
            }
        )
        df["model"] = m
        frames.append(df)
        log.info("CMIP6 %s %s klart (%d dagar)", resort.name, m, len(df))
        time.sleep(pause_s)
    out = pd.concat(frames, ignore_index=True)
    out["resort"] = resort.key
    out["grid_elevation"] = elevation
    return out, elevation


# ---------------------------------------------------------------------------
# Global temperatur (NASA GISTEMP)
# ---------------------------------------------------------------------------


def fetch_gistemp() -> pd.DataFrame:
    """Global anomali relativt 1951-1980, per år: helår (J-D) och vinter (DJF).

    DJF för år Y hos GISS betyder dec(Y-1)+jan(Y)+feb(Y), vilket råkar vara
    exakt samma vinteretikett som vi använder. Tack, NASA.
    """
    r = _SESSION.get(GISTEMP_URL, timeout=60)
    r.raise_for_status()
    df = pd.read_csv(pd.io.common.StringIO(r.text), skiprows=1, na_values=["***", "****"])
    df = df.rename(columns={"Year": "year", "J-D": "global_anom_annual", "DJF": "global_anom_djf"})
    return df[["year", "global_anom_annual", "global_anom_djf"]].astype(float)


In [ ]:
%%writefile skiclimate/store.py
"""Läs och skriv datamängder. Parquet om pyarrow finns, annars gzippad CSV.

Colab och Fabric har båda pyarrow. En naken Python på en Windowsburk kanske
inte har det, och då ska pipelinen ändå funka.
"""

from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd

log = logging.getLogger(__name__)

try:
    import pyarrow  # noqa: F401

    _HAVE_PARQUET = True
except ImportError:  # pragma: no cover
    _HAVE_PARQUET = False


def save(df: pd.DataFrame, path: Path) -> Path:
    """Sparar och returnerar den faktiska sökvägen (ändelsen kan bytas)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if _HAVE_PARQUET:
        target = path.with_suffix(".parquet")
        df.to_parquet(target, index=False)
    else:
        target = path.with_suffix(".csv.gz")
        df.to_csv(target, index=False, compression="gzip")
    log.info("Sparade %d rader -> %s", len(df), target)
    return target


def load(path: Path) -> pd.DataFrame | None:
    """Hittar filen oavsett vilken ändelse ``save`` råkade välja."""
    path = Path(path)
    for cand in (path.with_suffix(".parquet"), path.with_suffix(".csv.gz"), path):
        if cand.exists():
            if cand.suffix == ".parquet":
                return pd.read_parquet(cand)
            df = pd.read_csv(cand)
            for col in ("date", "first_obs", "last_obs"):
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col])
            return df
    return None


In [ ]:
%%writefile skiclimate/features.py
"""Från dygnsvärden till en rad per ort och vinter.

Steg 1: bygg en sammanhängande dygnsserie per ort. SMHI-stationerna är
        sanningen där de finns, ERA5 fyller luckorna (höjdkorrigerat).
Steg 2: räkna säsongsmått per vinter (juli-juni).

Alla trösklar ligger i ``config.SEASON``. Ändra dem där, inte här.
"""

from __future__ import annotations

import logging

import numpy as np
import pandas as pd

from .config import RESORTS, SEASON, SeasonRules

log = logging.getLogger(__name__)

DAILY_COLUMNS = ("t_mean", "t_min", "t_max", "precip", "snow_depth")


# ---------------------------------------------------------------------------
# Steg 1: en dygnsserie per ort
# ---------------------------------------------------------------------------


def _pick_station_series(obs: pd.DataFrame, stations: pd.DataFrame, resort: str, variable: str) -> pd.Series | None:
    """Slår ihop flera stationers värden till en serie: bästa stationen först,
    övriga fyller bara luckor.

    "Bäst" = längst serie, med avstånd som skiljedomare. En station som stod
    uppe 1961-2023 slår en som stått uppe sedan 2019, även om den nya är
    närmare. Trender kräver längd, inte närhet.
    """
    sub = obs[(obs["resort"] == resort) & (obs["variable"] == variable)]
    if sub.empty:
        return None
    st = stations[(stations["resort"] == resort) & (stations["variable"] == variable)].copy()
    st = st.sort_values(["n_obs", "distance_km"], ascending=[False, True])
    combined: pd.Series | None = None
    for key in st["station_key"]:
        s = sub[sub["station_key"] == key].set_index("date")["value"].sort_index()
        s = s[~s.index.duplicated()]
        combined = s if combined is None else combined.combine_first(s)
    return combined


def lapse_adjust(t: pd.Series, from_elevation: float, to_elevation: float, rules: SeasonRules = SEASON) -> pd.Series:
    """Flytta en temperaturserie i höjdled med fast lapse rate.

    Inversioner finns, men i ett dygnsmedel över 60 år jämnar de ut sig.
    Vill du ha inversioner får du hämta timdata och en radiosond.
    """
    if not np.isfinite(from_elevation) or not np.isfinite(to_elevation):
        return t
    return t - rules.lapse_rate * (to_elevation - from_elevation) / 100.0


def build_daily(
    resort_key: str,
    smhi_obs: pd.DataFrame | None,
    smhi_stations: pd.DataFrame | None,
    era5: pd.DataFrame | None,
    era5_snow: pd.DataFrame | None = None,
    target_elevation: int | None = None,
    rules: SeasonRules = SEASON,
) -> pd.DataFrame:
    """En rad per dag, kolumner enligt ``DAILY_COLUMNS`` plus ``source_*``-flaggor.

    ``target_elevation`` är höjden vi vill beskriva (byn som standard).
    Stationer lapse-justeras till den från sin egen höjd, ERA5 från sin gridhöjd.
    """
    resort = RESORTS[resort_key]
    target = target_elevation if target_elevation is not None else resort.base_elevation

    frames: dict[str, pd.Series] = {}
    sources: dict[str, pd.Series] = {}

    # --- ERA5 som botten
    if era5 is not None and not era5.empty:
        e = era5[era5["resort"] == resort_key].set_index("date").sort_index()
        e = e[~e.index.duplicated()]
        grid_z = float(e["grid_elevation"].iloc[0]) if "grid_elevation" in e else float("nan")
        for col in ("t_mean", "t_min", "t_max"):
            if col in e:
                frames[col] = lapse_adjust(e[col], grid_z, target, rules)
                sources[col] = pd.Series("era5", index=e.index)
        if "precip" in e:
            frames["precip"] = e["precip"]
            sources["precip"] = pd.Series("era5", index=e.index)
    if era5_snow is not None and not era5_snow.empty:
        s = era5_snow[era5_snow["resort"] == resort_key].set_index("date").sort_index()
        frames["snow_depth"] = s["snow_depth_era5_max_cm"]
        sources["snow_depth"] = pd.Series("era5", index=s.index)

    # --- SMHI-stationer ovanpå: där de finns vinner de
    if smhi_obs is not None and not smhi_obs.empty and smhi_stations is not None:
        for var in DAILY_COLUMNS:
            s = _pick_station_series(smhi_obs, smhi_stations, resort_key, var)
            if s is None or s.empty:
                continue
            if var.startswith("t_"):
                st = smhi_stations[(smhi_stations["resort"] == resort_key) & (smhi_stations["variable"] == var)]
                st = st.sort_values(["n_obs", "distance_km"], ascending=[False, True])
                z = float(st["height"].iloc[0]) if not st.empty else float("nan")
                # OBS: alla stationer får den bästa stationens höjd. Fel, men litet
                # fel, eftersom sekundärstationerna bara fyller luckor.
                s = lapse_adjust(s, z, target, rules)
            src = pd.Series("smhi", index=s.index)
            if var in frames:
                frames[var] = s.combine_first(frames[var])
                sources[var] = src.combine_first(sources[var])
            else:
                frames[var] = s
                sources[var] = src

    if not frames:
        raise ValueError(f"Ingen data alls för {resort.name}")

    daily = pd.DataFrame(frames)
    full_index = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    daily = daily.reindex(full_index)
    daily.index.name = "date"
    for var, src in sources.items():
        daily[f"source_{var}"] = src.reindex(full_index)

    # t_min/t_max saknas ibland hos stationerna (bara dygnsmedel mätt). Fyll
    # från t_mean med typisk dygnsamplitud i fjällen på vintern, ~3 °C åt varje håll.
    # Grov men bättre än NaN som dödar hela vintern.
    if "t_mean" in daily:
        if "t_min" not in daily:
            daily["t_min"] = daily["t_mean"] - 3.0
        else:
            daily["t_min"] = daily["t_min"].fillna(daily["t_mean"] - 3.0)
        if "t_max" not in daily:
            daily["t_max"] = daily["t_mean"] + 3.0
        else:
            daily["t_max"] = daily["t_max"].fillna(daily["t_mean"] + 3.0)

    # Korta luckor (≤3 dagar) i temperatur interpoleras. Längre får vara NaN.
    for col in ("t_mean", "t_min", "t_max"):
        if col in daily:
            daily[col] = daily[col].interpolate(limit=3, limit_area="inside")

    daily["resort"] = resort_key
    daily["target_elevation"] = target
    return daily.reset_index()


# ---------------------------------------------------------------------------
# Snömodell där snödjup saknas
# ---------------------------------------------------------------------------


def degree_day_snowpack(t_mean: np.ndarray, precip: np.ndarray, rules: SeasonRules = SEASON) -> np.ndarray:
    """Graddagsmodell. Snöar när det är kallt, smälter proportionellt mot plusgrader.

    Samma modell som webbsidans warming-vy, så siffrorna går att jämföra.
    Den vet inget om vind, sol eller att solen står 3° över horisonten i
    januari. Det är en bokföringsmodell, inte fysik. Räcker för trender.
    """
    n = len(t_mean)
    swe = np.zeros(n)  # mm vattenekvivalent i snötäcket
    cur = 0.0
    for i in range(n):
        t = t_mean[i]
        p = precip[i]
        if np.isnan(t):
            swe[i] = np.nan
            continue
        if not np.isnan(p) and t < rules.snow_temp_threshold:
            cur += p
        if t > 0:
            cur = max(0.0, cur - rules.melt_factor * t)
        swe[i] = cur
    # mm vatten -> cm snö: mm / (densitet/1000) / 10
    return swe / (rules.pack_density / 1000.0) / 10.0


# ---------------------------------------------------------------------------
# Steg 2: vintermått
# ---------------------------------------------------------------------------


def winter_year(dates: pd.Series, rules: SeasonRules = SEASON) -> pd.Series:
    """Vintern 2019/20 heter 2020: året januari ligger i."""
    d = pd.DatetimeIndex(dates)
    return pd.Series(np.where(d.month >= rules.year_start_month, d.year + 1, d.year), index=dates.index)


def _longest_run(mask: np.ndarray) -> int:
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


def _first_run_start(mask: np.ndarray, run: int) -> int | None:
    """Index där första sviten av ``run`` sanna värden börjar, annars None."""
    cur = 0
    for i, v in enumerate(mask):
        cur = cur + 1 if v else 0
        if cur >= run:
            return i - run + 1
    return None


def _last_run_end(mask: np.ndarray, run: int) -> int | None:
    rev = _first_run_start(mask[::-1], run)
    return None if rev is None else len(mask) - 1 - rev


def _in_months(month: np.ndarray, span: tuple[int, int]) -> np.ndarray:
    a, b = span
    return (month >= a) | (month <= b) if a > b else (month >= a) & (month <= b)


def winter_metrics(w: pd.DataFrame, rules: SeasonRules = SEASON) -> dict:
    """Alla mått för en enskild vinter (en DataFrame med dygnsrader jul-jun)."""
    w = w.sort_values("date")
    month = w["date"].dt.month.to_numpy()
    tm = w["t_mean"].to_numpy(dtype=float)
    tmin = w["t_min"].to_numpy(dtype=float)
    tmax = w["t_max"].to_numpy(dtype=float)
    p = w["precip"].to_numpy(dtype=float) if "precip" in w else np.full(len(w), np.nan)
    core = _in_months(month, rules.core_winter)
    making = _in_months(month, rules.making_season)
    n_days = len(w)
    valid_t = np.isfinite(tm)

    out: dict = {
        "n_days": n_days,
        "t_coverage": float(valid_t.mean()) if n_days else 0.0,
        "p_coverage": float(np.isfinite(p).mean()) if n_days else 0.0,
    }

    # --- temperatur
    out["t_mean_winter"] = float(np.nanmean(tm[core])) if core.any() else np.nan       # dec-mar
    out["t_mean_nov_apr"] = float(np.nanmean(tm[_in_months(month, (11, 4))]))
    out["frost_days"] = int(np.nansum(tmin < 0))
    out["ice_days"] = int(np.nansum(tmax < 0))
    out["thaw_days_core"] = int(np.nansum((tmax > rules.thaw_above) & core))
    out["snowmaking_days"] = int(np.nansum((tmin <= rules.snowmaking_tmin) & making))
    out["cold_sum"] = float(-np.nansum(np.minimum(tm, 0.0)))  # "frysgraddagar", hur mycket kyla vintern samlade
    out["warm_sum_core"] = float(np.nansum(np.maximum(tm[core], 0.0)))

    # --- termisk vinter: första/sista svit på run_length dygn under 0
    below = np.where(valid_t, tm < 0, False)
    start = _first_run_start(below, rules.run_length)
    end = _last_run_end(below, rules.run_length)
    out["thermal_winter_start_doy"] = int(start) if start is not None else np.nan  # dagar sedan 1 juli
    out["thermal_winter_end_doy"] = int(end) if end is not None else np.nan
    out["thermal_winter_length"] = (
        int(end - start + 1) if start is not None and end is not None else 0
    )

    # --- nederbörd
    out["precip_total"] = float(np.nansum(p)) if np.isfinite(p).any() else np.nan
    snow_mask = np.isfinite(p) & (tm < rules.snow_temp_threshold)
    out["snowfall_mm_we"] = float(np.nansum(p[snow_mask])) if np.isfinite(p).any() else np.nan
    out["snow_fraction"] = (
        out["snowfall_mm_we"] / out["precip_total"] if out.get("precip_total") and out["precip_total"] > 0 else np.nan
    )
    out["rain_on_snow_days"] = np.nan  # fylls i när vi vet snödjup

    # --- snödjup: observerat om det finns, annars graddagsmodell
    obs_depth = w["snow_depth"].to_numpy(dtype=float) if "snow_depth" in w else np.full(n_days, np.nan)
    depth_cov = float(np.isfinite(obs_depth).mean()) if n_days else 0.0
    if depth_cov >= 0.8:
        depth = pd.Series(obs_depth).interpolate(limit=5, limit_area="inside").to_numpy()
        out["snow_source"] = "observed"
    else:
        depth = degree_day_snowpack(tm, p, rules)
        out["snow_source"] = "modelled"
    out["snow_depth_coverage"] = depth_cov

    reliable = np.where(np.isfinite(depth), depth >= rules.reliable_depth_cm, False)
    cover = np.where(np.isfinite(depth), depth >= rules.cover_depth_cm, False)
    out["season_days_30cm"] = int(reliable.sum())
    out["season_core_30cm"] = int(_longest_run(reliable))
    out["cover_days"] = int(cover.sum())
    out["snow_reliable"] = bool(out["season_days_30cm"] >= rules.reliable_days)
    first = np.argmax(reliable) if reliable.any() else None
    last = (n_days - 1 - np.argmax(reliable[::-1])) if reliable.any() else None
    out["season_start_doy"] = int(first) if first is not None else np.nan
    out["season_end_doy"] = int(last) if last is not None else np.nan
    out["season_span"] = int(last - first + 1) if first is not None else 0
    out["max_depth_cm"] = float(np.nanmax(depth)) if np.isfinite(depth).any() else np.nan
    out["depth_feb1_cm"] = float(depth[(month == 2)][0]) if (month == 2).any() and np.isfinite(depth[(month == 2)]).any() else np.nan
    out["rain_on_snow_days"] = int(np.nansum((p >= 1.0) & (tm > rules.snow_temp_threshold) & cover))
    return out


def winter_table(daily: pd.DataFrame, rules: SeasonRules = SEASON, min_coverage: float = 0.9) -> pd.DataFrame:
    """En rad per (resort, vinter). Vintrar med för dålig täckning kastas."""
    df = daily.copy()
    df["winter"] = winter_year(df["date"], rules)
    rows = []
    for (resort, wy), w in df.groupby(["resort", "winter"]):
        if len(w) < 300:  # halvvintrar i början/slutet av serien är inte vintrar
            continue
        m = winter_metrics(w, rules)
        m["resort"] = resort
        m["winter"] = int(wy)
        rows.append(m)
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    bad = table["t_coverage"] < min_coverage
    if bad.any():
        log.info("Kastar %d vintrar med temperaturtäckning < %.0f%%", int(bad.sum()), min_coverage * 100)
    table = table[~bad]
    cols = ["resort", "winter"] + [c for c in table.columns if c not in ("resort", "winter")]
    return table[cols].sort_values(["resort", "winter"]).reset_index(drop=True)


def cmip6_winter_table(
    cmip6: pd.DataFrame,
    resort_key: str,
    target_elevation: int | None = None,
    rules: SeasonRules = SEASON,
) -> pd.DataFrame:
    """Samma vintermått, per klimatmodell. Snö alltid via graddagsmodell,
    så jämförelsen med observerade vintrar ska göras mot ``snow_source == modelled``
    eller via temperatur, inte via observerat snödjup."""
    resort = RESORTS[resort_key]
    target = target_elevation if target_elevation is not None else resort.base_elevation
    sub = cmip6[cmip6["resort"] == resort_key].copy()
    grid_z = float(sub["grid_elevation"].iloc[0]) if "grid_elevation" in sub and not sub.empty else float("nan")
    frames = []
    for model, m in sub.groupby("model"):
        d = m[["date", "t_mean", "t_min", "t_max", "precip"]].copy()
        for col in ("t_mean", "t_min", "t_max"):
            d[col] = lapse_adjust(d[col], grid_z, target, rules)
        d["resort"] = resort_key
        t = winter_table(d, rules)
        t["model"] = model
        frames.append(t)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [ ]:
%%writefile skiclimate/synthetic.py
"""Påhittad men trovärdig data, för att testa pipelinen utan nätverk.

Ingen av siffrorna här är en mätning. Det är ett årstidssinus, en linjär
uppvärmning, lite AR(1)-brus och en graddagsmodell. Om resultatet ser ut
som riktiga data är det för att riktiga data också mest är sinus och brus.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from .config import CMIP6_MODELS, RESORTS

#: ungefärlig årsmedeltemperatur i byn och amplitud, per ort
_CLIMATE = {
    "salen": {"mean": 3.0, "amp": 11.0, "precip_mm_day": 2.0},
    "are": {"mean": 2.5, "amp": 11.5, "precip_mm_day": 2.3},
    "tarnaby": {"mean": 0.8, "amp": 12.5, "precip_mm_day": 2.1},
}


def _daily_temp(dates: pd.DatetimeIndex, mean: float, amp: float, trend_per_year: float, rng: np.random.Generator, start_year: int) -> np.ndarray:
    doy = dates.dayofyear.to_numpy()
    years = (dates.year + doy / 365.25).to_numpy() - start_year
    seasonal = mean - amp * np.cos(2 * np.pi * (doy - 15) / 365.25)  # kallast runt 15 jan
    # AR(1) med phi 0.8: vädret minns gårdagen, inte förra veckan
    noise = np.empty(len(dates))
    noise[0] = rng.normal(0, 3)
    eps = rng.normal(0, 2.0, len(dates))
    for i in range(1, len(dates)):
        noise[i] = 0.8 * noise[i - 1] + eps[i]
    return seasonal + trend_per_year * years + noise


def synthetic_era5(resort_key: str, start: str = "1961-01-01", end: str = "2025-06-30", seed: int = 0, trend_per_year: float = 0.04) -> pd.DataFrame:
    rng = np.random.default_rng(seed + hash(resort_key) % 1000)
    r = RESORTS[resort_key]
    c = _CLIMATE[resort_key]
    dates = pd.date_range(start, end, freq="D")
    tm = _daily_temp(dates, c["mean"], c["amp"], trend_per_year, rng, pd.Timestamp(start).year)
    precip = rng.gamma(0.7, c["precip_mm_day"] / 0.7, len(dates)) * (rng.random(len(dates)) < 0.55)
    df = pd.DataFrame(
        {
            "date": dates,
            "t_mean": tm,
            "t_min": tm - rng.uniform(2, 5, len(dates)),
            "t_max": tm + rng.uniform(2, 5, len(dates)),
            "precip": precip,
            "rain": np.where(tm > 1, precip, 0.0),
            "snowfall_cm": np.where(tm <= 1, precip * 1.0, 0.0),
            "wind_max": rng.gamma(4, 2, len(dates)),
            "radiation": np.clip(10 + 12 * np.sin(2 * np.pi * (dates.dayofyear - 80) / 365.25), 0, None),
            "resort": resort_key,
            "source": "synthetic",
            "grid_elevation": float(r.base_elevation + 150),
        }
    )
    return df


def synthetic_smhi(resort_key: str, era5: pd.DataFrame, seed: int = 1) -> tuple[pd.DataFrame, pd.DataFrame]:
    """En 'station' i byn med snödjup från graddagsmodellen + brus, temperatur = ERA5 + brus.

    Stationen saknar 8 % av dagarna slumpmässigt och ett helt år (1975), så
    att lucklogiken faktiskt får jobba.
    """
    from .features import degree_day_snowpack  # sen import, annars cirkel

    rng = np.random.default_rng(seed)
    r = RESORTS[resort_key]
    e = era5.set_index("date")
    depth = degree_day_snowpack(e["t_mean"].to_numpy(), e["precip"].to_numpy())
    depth = np.clip(depth * rng.normal(1.0, 0.15, len(depth)) + rng.normal(0, 1.5, len(depth)), 0, None)
    variables = {
        2: ("t_mean", e["t_mean"] + rng.normal(0, 0.5, len(e))),
        19: ("t_min", e["t_min"] + rng.normal(0, 0.5, len(e))),
        20: ("t_max", e["t_max"] + rng.normal(0, 0.5, len(e))),
        5: ("precip", e["precip"] * rng.normal(1.0, 0.1, len(e))),
        8: ("snow_depth", pd.Series(depth, index=e.index)),
    }
    keep = rng.random(len(e)) > 0.08
    keep &= e.index.year != 1975
    obs, st = [], []
    station_key = 100000 + abs(hash(resort_key)) % 9000
    for pid, (var, series) in variables.items():
        s = series[keep]
        obs.append(
            pd.DataFrame(
                {"date": s.index, "value": s.to_numpy(), "quality": "G", "resort": resort_key,
                 "parameter": pid, "variable": var, "unit": "?", "station_key": station_key}
            )
        )
        st.append(
            {"resort": resort_key, "parameter": pid, "variable": var, "station_key": station_key,
             "name": f"{r.name} (syntetisk)", "lat": r.lat, "lon": r.lon, "height": float(r.base_elevation),
             "distance_km": 0.5, "active": True, "from_year": s.index.min().year, "to_year": s.index.max().year,
             "n_obs": len(s), "first_obs": s.index.min(), "last_obs": s.index.max()}
        )
    return pd.concat(obs, ignore_index=True), pd.DataFrame(st)


def synthetic_cmip6(resort_key: str, seed: int = 2) -> pd.DataFrame:
    """Sju 'modeller' 1950-2050 med olika klimatkänslighet. SSP5-8.5-ish: 0.05-0.08 °C/år."""
    r = RESORTS[resort_key]
    c = _CLIMATE[resort_key]
    frames = []
    for i, m in enumerate(CMIP6_MODELS):
        rng = np.random.default_rng(seed * 100 + i + hash(resort_key) % 100)
        dates = pd.date_range("1950-01-01", "2050-12-31", freq="D")
        trend = 0.05 + 0.005 * i
        tm = _daily_temp(dates, c["mean"] - 0.3, c["amp"], trend, rng, 1950)
        precip = rng.gamma(0.7, c["precip_mm_day"] / 0.7, len(dates)) * (rng.random(len(dates)) < 0.55)
        frames.append(
            pd.DataFrame(
                {"date": dates, "t_mean": tm, "t_min": tm - 3.5, "t_max": tm + 3.5, "precip": precip,
                 "snowfall_cm": np.where(tm <= 1, precip, 0.0), "model": m, "resort": resort_key,
                 "grid_elevation": float(r.base_elevation + 200)}
            )
        )
    return pd.concat(frames, ignore_index=True)


def synthetic_gistemp(start: int = 1880, end: int = 2025, seed: int = 3) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    years = np.arange(start, end + 1)
    # ungefär den riktiga kurvan: platt till 1970, sedan ~0.02 °C/år
    anom = np.where(years < 1970, -0.1, -0.1 + 0.02 * (years - 1970)) + rng.normal(0, 0.08, len(years))
    return pd.DataFrame({"year": years, "global_anom_annual": anom, "global_anom_djf": anom + rng.normal(0, 0.1, len(years))})


In [ ]:
%%writefile skiclimate/analysis.py
"""Statistik och ML på vintertabellen.

Fyra frågor, fyra verktyg:

1. Finns det en trend?            -> Theil-Sen + Mann-Kendall, OLS med HAC-fel
2. När bröt det?                  -> en brytpunkt i medelnivå, bootstrappad
3. Hur känslig är säsongen?       -> säsongsdagar per °C vintertemperatur, och
                                     lokal vintertemperatur per °C global
4. Hur långa blir säsongerna?     -> gradient boosting tränad på observationer,
                                     applicerad på biaskorrigerade CMIP6-vintrar

Inget här är en prognos för nästa vinter. Det är klimat, inte väder.
"""

from __future__ import annotations

import logging
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
from scipy import stats

from .config import PERIODS

log = logging.getLogger(__name__)

#: mått värda att trenda. Ordning = ordning i rapporten.
TREND_METRICS = (
    "season_days_30cm",
    "season_core_30cm",
    "cover_days",
    "thermal_winter_length",
    "t_mean_winter",
    "frost_days",
    "thaw_days_core",
    "snowmaking_days",
    "max_depth_cm",
    "rain_on_snow_days",
    "snow_fraction",
)


# ---------------------------------------------------------------------------
# 1. Trender
# ---------------------------------------------------------------------------


def mann_kendall(x: np.ndarray) -> tuple[float, float, float]:
    """Mann-Kendall-test för monoton trend. Returnerar (S, z, p).

    Icke-parametriskt: bryr sig inte om fördelningen, bara om senare värden
    tenderar att vara större än tidigare. Tie-korrigerad varians enligt
    Kendall (1975). Ingen autokorrelationskorrigering – vintrar är hyfsat
    oberoende av varandra, till skillnad från dagar.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 4:
        return np.nan, np.nan, np.nan
    s = 0.0
    for i in range(n - 1):
        s += np.sign(x[i + 1 :] - x[i]).sum()
    _, counts = np.unique(x, return_counts=True)
    tie_term = np.sum(counts * (counts - 1) * (2 * counts + 5))
    var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18.0
    if var_s <= 0:
        return s, 0.0, 1.0
    z = (s - 1) / np.sqrt(var_s) if s > 0 else (s + 1) / np.sqrt(var_s) if s < 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(s), float(z), float(p)


@dataclass
class TrendResult:
    resort: str
    metric: str
    n: int
    first_year: int
    last_year: int
    mean: float
    #: Theil-Sen-lutning per decennium, med 95 % konfidensintervall
    sen_slope_per_decade: float
    sen_lo: float
    sen_hi: float
    #: Mann-Kendall p-värde
    mk_p: float
    #: OLS-lutning per decennium med Newey-West-standardfel (lag 2) och p
    ols_slope_per_decade: float
    ols_se: float
    ols_p: float
    #: total förändring över perioden enligt Theil-Sen
    total_change: float


def trend(series: pd.Series, years: pd.Series, resort: str, metric: str) -> TrendResult | None:
    ok = np.isfinite(series.to_numpy(dtype=float))
    y = series.to_numpy(dtype=float)[ok]
    t = years.to_numpy(dtype=float)[ok]
    if len(y) < 10:
        return None
    sen = stats.theilslopes(y, t, alpha=0.95)
    _, _, mk_p = mann_kendall(y)

    import statsmodels.api as sm  # tungt att importera, gör det bara här

    X = sm.add_constant(t)
    ols = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
    return TrendResult(
        resort=resort,
        metric=metric,
        n=int(len(y)),
        first_year=int(t.min()),
        last_year=int(t.max()),
        mean=float(y.mean()),
        sen_slope_per_decade=float(sen.slope * 10),
        sen_lo=float(sen.low_slope * 10),
        sen_hi=float(sen.high_slope * 10),
        mk_p=float(mk_p),
        ols_slope_per_decade=float(ols.params[1] * 10),
        ols_se=float(ols.bse[1] * 10),
        ols_p=float(ols.pvalues[1]),
        total_change=float(sen.slope * (t.max() - t.min())),
    )


def all_trends(winters: pd.DataFrame, metrics: tuple[str, ...] = TREND_METRICS) -> pd.DataFrame:
    rows = []
    for resort, g in winters.groupby("resort"):
        for m in metrics:
            if m not in g:
                continue
            r = trend(g[m], g["winter"], resort, m)
            if r is not None:
                rows.append(asdict(r))
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 2. Brytpunkt
# ---------------------------------------------------------------------------


def _best_split(y: np.ndarray, min_seg: int) -> tuple[int, float]:
    """Index och SSE-minskning för bästa enkla nivåskifte."""
    n = len(y)
    total = ((y - y.mean()) ** 2).sum()
    best_k, best_gain = -1, 0.0
    cs = np.cumsum(y)
    cs2 = np.cumsum(y**2)
    for k in range(min_seg, n - min_seg + 1):
        left = cs2[k - 1] - cs[k - 1] ** 2 / k
        right = (cs2[-1] - cs2[k - 1]) - (cs[-1] - cs[k - 1]) ** 2 / (n - k)
        gain = total - left - right
        if gain > best_gain:
            best_k, best_gain = k, gain
    return best_k, best_gain


def changepoint(series: pd.Series, years: pd.Series, min_seg: int = 10, n_boot: int = 999, seed: int = 0) -> dict:
    """En brytpunkt i medelnivå, p-värde via permutation.

    Nollhypotes: ingen brytpunkt, vintrarna är utbytbara. Vi blandar om
    serien 999 gånger och ser hur ofta slumpen hittar ett lika bra skifte.
    Bara medelnivå – en gradvis trend kan också ge en "brytpunkt" i mitten,
    så läs den ihop med trendtestet, inte istället för.
    """
    ok = np.isfinite(series.to_numpy(dtype=float))
    y = series.to_numpy(dtype=float)[ok]
    t = years.to_numpy()[ok]
    if len(y) < 2 * min_seg + 2:
        return {"year": np.nan, "before": np.nan, "after": np.nan, "shift": np.nan, "p": np.nan}
    k, gain = _best_split(y, min_seg)
    rng = np.random.default_rng(seed)
    hits = 0
    for _ in range(n_boot):
        _, g = _best_split(rng.permutation(y), min_seg)
        hits += g >= gain
    p = (hits + 1) / (n_boot + 1)
    return {
        "year": int(t[k]),
        "before": float(y[:k].mean()),
        "after": float(y[k:].mean()),
        "shift": float(y[k:].mean() - y[:k].mean()),
        "p": float(p),
    }


def all_changepoints(winters: pd.DataFrame, metrics: tuple[str, ...] = ("season_days_30cm", "t_mean_winter", "thermal_winter_length")) -> pd.DataFrame:
    rows = []
    for resort, g in winters.groupby("resort"):
        for m in metrics:
            if m in g:
                r = changepoint(g[m], g["winter"])
                r.update(resort=resort, metric=m)
                rows.append(r)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Periodjämförelse (1961-1990 mot 1991-2020)
# ---------------------------------------------------------------------------


def period_comparison(winters: pd.DataFrame, metrics: tuple[str, ...] = TREND_METRICS) -> pd.DataFrame:
    a0, a1 = PERIODS["past"]
    b0, b1 = PERIODS["present"]
    rows = []
    for resort, g in winters.groupby("resort"):
        past = g[(g["winter"] >= a0) & (g["winter"] <= a1)]
        pres = g[(g["winter"] >= b0) & (g["winter"] <= b1)]
        for m in metrics:
            if m not in g:
                continue
            x, y = past[m].dropna(), pres[m].dropna()
            if len(x) < 5 or len(y) < 5:
                continue
            welch = stats.ttest_ind(x, y, equal_var=False)
            mw = stats.mannwhitneyu(x, y, alternative="two-sided")
            rows.append(
                {
                    "resort": resort, "metric": m,
                    "past_mean": float(x.mean()), "present_mean": float(y.mean()),
                    "diff": float(y.mean() - x.mean()), "n_past": len(x), "n_present": len(y),
                    "welch_p": float(welch.pvalue), "mannwhitney_p": float(mw.pvalue),
                }
            )
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 3. Känslighet
# ---------------------------------------------------------------------------


def season_sensitivity(winters: pd.DataFrame, target: str = "season_days_30cm") -> pd.DataFrame:
    """Säsongsdagar per °C vintertemperatur (och per 100 mm nederbörd), per ort.

    OLS med två prediktorer. Koefficienten på temperatur är den siffra alla
    egentligen vill ha: "en grad varmare = så här många dagar kortare".
    """
    import statsmodels.api as sm

    rows = []
    for resort, g in winters.groupby("resort"):
        d = g[[target, "t_mean_winter", "precip_total"]].dropna()
        if len(d) < 15:
            continue
        X = sm.add_constant(np.column_stack([d["t_mean_winter"], d["precip_total"] / 100.0]))
        fit = sm.OLS(d[target].to_numpy(), X).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
        rows.append(
            {
                "resort": resort, "target": target, "n": len(d),
                "days_per_degC": float(fit.params[1]), "days_per_degC_se": float(fit.bse[1]), "p_temp": float(fit.pvalues[1]),
                "days_per_100mm": float(fit.params[2]), "p_precip": float(fit.pvalues[2]),
                "r2": float(fit.rsquared),
            }
        )
    return pd.DataFrame(rows)


def global_coupling(winters: pd.DataFrame, gistemp: pd.DataFrame | None) -> pd.DataFrame:
    """Lokal vintertemperatur mot global anomali: lutning = förstärkningsfaktor.

    Skandinaviska vintrar värms ungefär dubbelt så fort som globen. Om vi
    får ~2 här är det ett tecken på att datat inte är trasigt.
    """
    if gistemp is None or gistemp.empty:
        return pd.DataFrame()
    import statsmodels.api as sm

    rows = []
    for resort, g in winters.groupby("resort"):
        d = g.merge(gistemp, left_on="winter", right_on="year", how="inner")
        d = d[["t_mean_winter", "global_anom_djf", "season_days_30cm"]].dropna()
        if len(d) < 15:
            continue
        X = sm.add_constant(d["global_anom_djf"].to_numpy())
        f_t = sm.OLS(d["t_mean_winter"].to_numpy(), X).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
        f_s = sm.OLS(d["season_days_30cm"].to_numpy(), X).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
        rows.append(
            {
                "resort": resort, "n": len(d),
                "local_degC_per_global_degC": float(f_t.params[1]), "se": float(f_t.bse[1]), "p": float(f_t.pvalues[1]), "r2_temp": float(f_t.rsquared),
                "season_days_per_global_degC": float(f_s.params[1]), "p_season": float(f_s.pvalues[1]), "r2_season": float(f_s.rsquared),
            }
        )
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 4. Projektion
# ---------------------------------------------------------------------------

#: prediktorer som finns både i observerade vintrar och i CMIP6-vintrar
PROJECTION_FEATURES = (
    "t_mean_winter",
    "t_mean_nov_apr",
    "frost_days",
    "thaw_days_core",
    "snowmaking_days",
    "cold_sum",
    "warm_sum_core",
    "precip_total",
    "snowfall_mm_we",
)


def bias_correct(cmip6_winters: pd.DataFrame, obs_winters: pd.DataFrame, features: tuple[str, ...] = PROJECTION_FEATURES, ref: tuple[int, int] = PERIODS["present"]) -> pd.DataFrame:
    """Delta-korrigering per modell och ort: skjut varje modells fördelning så
    att medel och spridning matchar observationerna under referensperioden.

    Open-Meteo har redan biaskorrigerat mot ERA5, men ERA5:s gridruta är inte
    vår station. Ett andra lager är billigt och gör siffrorna jämförbara.
    Kvantilmappning vore finare; med 30 referensvintrar är det mest brus.
    """
    out = []
    a, b = ref
    for (resort, model), g in cmip6_winters.groupby(["resort", "model"]):
        g = g.copy()
        o = obs_winters[(obs_winters["resort"] == resort) & (obs_winters["winter"].between(a, b))]
        m = g[g["winter"].between(a, b)]
        if len(o) < 10 or len(m) < 10:
            out.append(g)
            continue
        for f in features:
            if f not in g or f not in o:
                continue
            o_mu, o_sd = o[f].mean(), o[f].std(ddof=1)
            m_mu, m_sd = m[f].mean(), m[f].std(ddof=1)
            scale = (o_sd / m_sd) if m_sd and np.isfinite(m_sd) and m_sd > 1e-9 else 1.0
            g[f] = o_mu + (g[f] - m_mu) * scale
            # räknade dagar kan inte bli negativa hur mycket vi än korrigerar
            if f.endswith("_days") or f.endswith("_sum") or f in ("precip_total", "snowfall_mm_we"):
                g[f] = g[f].clip(lower=0)
        out.append(g)
    return pd.concat(out, ignore_index=True)


@dataclass
class ProjectionModel:
    resort: str
    target: str
    cv_mae_gbr: float
    cv_mae_linear: float
    cv_mae_climatology: float
    chosen: str
    n_train: int


def _cv_mae(model_factory, X: np.ndarray, y: np.ndarray, n_splits: int = 5) -> float:
    from sklearn.model_selection import TimeSeriesSplit

    tss = TimeSeriesSplit(n_splits=n_splits)
    errs = []
    for tr, te in tss.split(X):
        m = model_factory()
        m.fit(X[tr], y[tr])
        errs.append(np.abs(m.predict(X[te]) - y[te]).mean())
    return float(np.mean(errs))


def fit_projection(
    obs_winters: pd.DataFrame,
    cmip6_winters: pd.DataFrame,
    target: str = "season_days_30cm",
    features: tuple[str, ...] = PROJECTION_FEATURES,
    seed: int = 0,
) -> tuple[pd.DataFrame, list[ProjectionModel]]:
    """Tränar per ort, väljer den modell som vinner tidsserie-CV, och kör den
    på varje CMIP6-modells vintrar. Returnerar (projektioner, modellsammanfattning).

    Kandidater:
      * HistGradientBoosting (icke-linjär, hanterar trösklar i snö bra)
      * Ridge (linjär, tråkig, svårslagen på 60 datapunkter)
      * klimatologi (medelvärdet – om vi inte slår den ska vi skämmas)

    Osäkerhet: kvantil-GBR för p10/p50/p90 ovanpå vinnaren, plus spridningen
    mellan de sju klimatmodellerna. Den senare är oftast större.
    """
    from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    feats = [f for f in features if f in obs_winters and f in cmip6_winters]
    projections, summaries = [], []

    for resort, g in obs_winters.groupby("resort"):
        d = g[[*feats, target, "winter"]].dropna().sort_values("winter")
        if len(d) < 20:
            log.warning("%s: bara %d vintrar, hoppar projektion", resort, len(d))
            continue
        X, y = d[feats].to_numpy(dtype=float), d[target].to_numpy(dtype=float)

        gbr_factory = lambda: HistGradientBoostingRegressor(  # noqa: E731
            max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=5, l2_regularization=1.0, random_state=seed
        )
        lin_factory = lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0))  # noqa: E731

        class _Clim:
            def fit(self, X, y):
                self.mu = y.mean()
                return self

            def predict(self, X):
                return np.full(len(X), self.mu)

        mae = {
            "gbr": _cv_mae(gbr_factory, X, y),
            "linear": _cv_mae(lin_factory, X, y),
            "climatology": _cv_mae(_Clim, X, y),
        }
        chosen = min(("gbr", "linear"), key=mae.get)
        if mae[chosen] >= mae["climatology"]:
            log.warning("%s: ingen modell slår klimatologin (MAE %.1f). Säsongsprojektionen är värdelös här.", resort, mae["climatology"])
        summaries.append(ProjectionModel(resort, target, mae["gbr"], mae["linear"], mae["climatology"], chosen, len(d)))

        point = (gbr_factory() if chosen == "gbr" else lin_factory()).fit(X, y)
        quantiles = {}
        for q in (0.1, 0.5, 0.9):
            quantiles[q] = GradientBoostingRegressor(
                loss="quantile", alpha=q, n_estimators=300, max_depth=2, learning_rate=0.05, min_samples_leaf=5, random_state=seed
            ).fit(X, y)

        c = cmip6_winters[cmip6_winters["resort"] == resort].dropna(subset=feats)
        if c.empty:
            continue
        Xc = c[feats].to_numpy(dtype=float)
        pred = c[["resort", "model", "winter"]].copy()
        pred["pred"] = np.clip(point.predict(Xc), 0, None)
        for q, m in quantiles.items():
            pred[f"p{int(q * 100)}"] = np.clip(m.predict(Xc), 0, None)
        pred["target"] = target
        projections.append(pred)

    return (pd.concat(projections, ignore_index=True) if projections else pd.DataFrame()), summaries


def summarise_projection(proj: pd.DataFrame, reliable_days: int = 100) -> pd.DataFrame:
    """Per ort och decennium: ensemblemedian, spridning mellan modeller, och
    andelen modellvintrar som klarar 100-dagarsregeln."""
    if proj.empty:
        return proj
    p = proj.copy()
    p["decade"] = (p["winter"] // 10) * 10
    rows = []
    for (resort, dec), g in p.groupby(["resort", "decade"]):
        per_model = g.groupby("model")["pred"].mean()
        rows.append(
            {
                "resort": resort, "decade": int(dec), "n_models": int(per_model.size),
                "median": float(per_model.median()), "min_model": float(per_model.min()), "max_model": float(per_model.max()),
                "p10": float(g["p10"].mean()), "p90": float(g["p90"].mean()),
                "share_reliable": float((g["pred"] >= reliable_days).mean()),
            }
        )
    return pd.DataFrame(rows)


def first_unreliable_decade(summary: pd.DataFrame, threshold: float = 0.5, from_decade: int = 2020) -> pd.DataFrame:
    """Första decenniet från ``from_decade`` där under hälften av modellvintrarna
    klarar 100 dagar. ``already`` säger om orten redan låg under vid startdecenniet –
    då är frågan inte "när" utan "sedan när", och den svarar observationerna på."""
    rows = []
    for resort, g in summary.groupby("resort"):
        g = g[g["decade"] >= from_decade].sort_values("decade")
        hit = g[g["share_reliable"] < threshold]
        first = int(hit["decade"].iloc[0]) if not hit.empty else None
        rows.append({"resort": resort, "first_unreliable_decade": first, "already": bool(first == from_decade)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile skiclimate/report.py
"""Diagram och en markdown-sammanfattning. Inget interaktivt – detta är PNG:er
man kan klistra in i en rapport eller visa i en notebook.

Designregler som följs: en ort = en fast färg, en y-axel per diagram, tunna
linjer, ingen regnbåge, ingen sekundäraxel. Textfärg är alltid textfärg.
"""

from __future__ import annotations

import logging
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # ingen skärm på servern, och ingen i Fabric heller
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from scipy import stats  # noqa: E402

from .config import PERIODS, RESORTS, RESORT_COLORS, SEASON  # noqa: E402

log = logging.getLogger(__name__)

_TEXT = "#0b0b0b"
_MUTED = "#52514e"
_GRID = "#e6e5e1"
_SURFACE = "#fcfcfb"

METRIC_LABELS = {
    "season_days_30cm": "Dagar med ≥30 cm snö",
    "season_core_30cm": "Längsta sammanhängande period ≥30 cm (dagar)",
    "cover_days": "Dagar med snötäcke",
    "thermal_winter_length": "Termisk vinter (dagar)",
    "t_mean_winter": "Medeltemperatur dec–mar (°C)",
    "frost_days": "Frostnätter",
    "thaw_days_core": "Tödagar dec–mar",
    "snowmaking_days": "Snökanondagar nov–mar",
    "max_depth_cm": "Största snödjup (cm)",
    "rain_on_snow_days": "Regn-på-snö-dagar",
    "snow_fraction": "Andel nederbörd som snö",
}


def _style(ax: plt.Axes, ylabel: str = "") -> None:
    ax.set_facecolor(_SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(_GRID)
    ax.grid(axis="y", color=_GRID, linewidth=0.8)
    ax.tick_params(colors=_MUTED, labelsize=9)
    ax.set_ylabel(ylabel, color=_MUTED, fontsize=9)


def _resort_label(key: str) -> str:
    return RESORTS[key].name if key in RESORTS else key


def plot_metric_trends(winters: pd.DataFrame, metric: str, out: Path) -> Path:
    """Små multiplar: en panel per ort, årsvärden som punkter, Theil-Sen-linje,
    och 1961–1990 / 1991–2020-medel som två horisontella streck."""
    resorts = [r for r in RESORTS if r in set(winters["resort"])]
    fig, axes = plt.subplots(1, len(resorts), figsize=(4.2 * len(resorts), 3.6), sharey=True, facecolor=_SURFACE)
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, resorts):
        g = winters[winters["resort"] == r].dropna(subset=[metric]).sort_values("winter")
        color = RESORT_COLORS.get(r, "#2a78d6")
        ax.plot(g["winter"], g[metric], color=color, linewidth=1.2, alpha=0.6)
        ax.scatter(g["winter"], g[metric], color=color, s=12, zorder=3)
        if len(g) >= 10:
            sen = stats.theilslopes(g[metric], g["winter"])
            xs = np.array([g["winter"].min(), g["winter"].max()])
            ax.plot(xs, sen.intercept + sen.slope * xs, color=_TEXT, linewidth=2)
            ax.text(
                0.02, 0.96, f"{sen.slope * 10:+.1f} per decennium", transform=ax.transAxes,
                fontsize=9, color=_TEXT, va="top",
            )
        for pid, (a, b) in (("past", PERIODS["past"]), ("present", PERIODS["present"])):
            m = g[g["winter"].between(a, b)][metric].mean()
            if np.isfinite(m):
                ax.hlines(m, a, b, color=_MUTED, linewidth=1, linestyle="--")
        ax.set_title(_resort_label(r), color=_TEXT, fontsize=11, loc="left")
        _style(ax, METRIC_LABELS.get(metric, metric) if ax is axes[0] else "")
    fig.suptitle(METRIC_LABELS.get(metric, metric), color=_TEXT, fontsize=12, x=0.01, ha="left")
    fig.tight_layout()
    path = out / f"trend_{metric}.png"
    fig.savefig(path, dpi=150, facecolor=_SURFACE)
    plt.close(fig)
    return path


def plot_sensitivity(winters: pd.DataFrame, out: Path, target: str = "season_days_30cm") -> Path:
    """Säsongsdagar mot vintertemperatur, en regressionslinje per ort."""
    fig, ax = plt.subplots(figsize=(7, 4.5), facecolor=_SURFACE)
    for r in RESORTS:
        g = winters[winters["resort"] == r].dropna(subset=[target, "t_mean_winter"])
        if g.empty:
            continue
        color = RESORT_COLORS[r]
        ax.scatter(g["t_mean_winter"], g[target], color=color, s=16, alpha=0.7, label=_resort_label(r))
        if len(g) >= 10:
            slope, intercept = np.polyfit(g["t_mean_winter"], g[target], 1)
            xs = np.linspace(g["t_mean_winter"].min(), g["t_mean_winter"].max(), 20)
            ax.plot(xs, intercept + slope * xs, color=color, linewidth=2)
            ax.text(xs[-1], intercept + slope * xs[-1], f" {slope:+.0f} d/°C", color=_TEXT, fontsize=9, va="center")
    ax.axhline(SEASON.reliable_days, color=_MUTED, linestyle="--", linewidth=1)
    ax.text(ax.get_xlim()[1], SEASON.reliable_days, "100-dagarsregeln ", color=_MUTED, fontsize=8, va="bottom", ha="right")
    ax.set_xlabel("Medeltemperatur dec–mar (°C)", color=_MUTED, fontsize=9)
    _style(ax, METRIC_LABELS.get(target, target))
    ax.legend(frameon=False, fontsize=9, labelcolor=_TEXT)
    ax.set_title("Hur mycket säsong kostar en grad?", color=_TEXT, fontsize=12, loc="left")
    fig.tight_layout()
    path = out / "sensitivity.png"
    fig.savefig(path, dpi=150, facecolor=_SURFACE)
    plt.close(fig)
    return path


def plot_projection(winters: pd.DataFrame, proj: pd.DataFrame, out: Path, target: str = "season_days_30cm") -> Path | None:
    """Observerat till idag, sedan ensemblemedian med modellspridning som band.

    Bandet är min–max mellan de sju modellerna (10-års glidande), inte ett
    konfidensintervall. Sju modeller är inte ett stickprov ur något.
    """
    if proj.empty:
        return None
    resorts = [r for r in RESORTS if r in set(proj["resort"])]
    fig, axes = plt.subplots(1, len(resorts), figsize=(4.2 * len(resorts), 3.8), sharey=True, facecolor=_SURFACE)
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, resorts):
        color = RESORT_COLORS[r]
        o = winters[winters["resort"] == r].dropna(subset=[target]).sort_values("winter")
        ax.plot(o["winter"], o[target].rolling(10, center=True, min_periods=5).mean(), color=color, linewidth=2, label="observerat, 10-årsmedel")
        ax.scatter(o["winter"], o[target], color=color, s=8, alpha=0.4)
        p = proj[proj["resort"] == r].pivot_table(index="winter", columns="model", values="pred")
        p = p.rolling(10, center=True, min_periods=5).mean()
        ax.fill_between(p.index, p.min(axis=1), p.max(axis=1), color=color, alpha=0.15, linewidth=0, label="spridning, 7 CMIP6-modeller")
        ax.plot(p.index, p.median(axis=1), color=_TEXT, linewidth=1.5, linestyle="--", label="ensemblemedian")
        ax.axhline(SEASON.reliable_days, color=_MUTED, linewidth=1, linestyle=":")
        ax.axvline(o["winter"].max(), color=_GRID, linewidth=1)
        ax.set_title(_resort_label(r), color=_TEXT, fontsize=11, loc="left")
        _style(ax, METRIC_LABELS.get(target, target) if ax is axes[0] else "")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, fontsize=8, labelcolor=_TEXT, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle("Projicerad säsongslängd, SSP5-8.5 (varmaste scenariot)", color=_TEXT, fontsize=12, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    path = out / f"projection_{target}.png"
    fig.savefig(path, dpi=150, facecolor=_SURFACE)
    plt.close(fig)
    return path


def plot_global_coupling(winters: pd.DataFrame, gistemp: pd.DataFrame | None, out: Path) -> Path | None:
    if gistemp is None or gistemp.empty:
        return None
    fig, ax = plt.subplots(figsize=(7, 4.5), facecolor=_SURFACE)
    for r in RESORTS:
        g = winters[winters["resort"] == r].merge(gistemp, left_on="winter", right_on="year").dropna(subset=["t_mean_winter", "global_anom_djf"])
        if len(g) < 10:
            continue
        color = RESORT_COLORS[r]
        ax.scatter(g["global_anom_djf"], g["t_mean_winter"], color=color, s=16, alpha=0.7, label=_resort_label(r))
        slope, intercept = np.polyfit(g["global_anom_djf"], g["t_mean_winter"], 1)
        xs = np.linspace(g["global_anom_djf"].min(), g["global_anom_djf"].max(), 20)
        ax.plot(xs, intercept + slope * xs, color=color, linewidth=2)
        ax.text(xs[-1], intercept + slope * xs[-1], f" ×{slope:.1f}", color=_TEXT, fontsize=9, va="center")
    ax.set_xlabel("Global vinteranomali DJF, GISTEMP (°C mot 1951–1980)", color=_MUTED, fontsize=9)
    _style(ax, "Lokal medeltemperatur dec–mar (°C)")
    ax.legend(frameon=False, fontsize=9, labelcolor=_TEXT)
    ax.set_title("Lokal vinter mot global uppvärmning", color=_TEXT, fontsize=12, loc="left")
    fig.tight_layout()
    path = out / "global_coupling.png"
    fig.savefig(path, dpi=150, facecolor=_SURFACE)
    plt.close(fig)
    return path


# ---------------------------------------------------------------------------
# Markdown
# ---------------------------------------------------------------------------


def _fmt(x, nd=1) -> str:
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "–"
    return f"{x:.{nd}f}"


def _sig(p: float) -> str:
    if not np.isfinite(p):
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def write_summary(
    out: Path,
    winters: pd.DataFrame,
    trends: pd.DataFrame,
    changepoints: pd.DataFrame,
    periods: pd.DataFrame,
    sensitivity: pd.DataFrame,
    coupling: pd.DataFrame,
    proj_summary: pd.DataFrame,
    unreliable: pd.DataFrame,
    model_notes: list,
    figures: list[Path],
    data_note: str = "",
) -> Path:
    L: list[str] = []
    L.append("# Skidorternas vintrar – Sälen, Åre, Tärnaby\n")
    if data_note:
        L.append(f"> {data_note}\n")
    L.append("Vinter *N* = juli *N−1* till juni *N*. Signifikans: \\* p<0,05, \\*\\* p<0,01, \\*\\*\\* p<0,001 (Mann-Kendall).\n")

    L.append("## Datatäckning\n")
    L.append("| Ort | Vintrar | Första | Sista | Snödjup observerat (andel vintrar) |")
    L.append("|---|---|---|---|---|")
    for r, g in winters.groupby("resort"):
        obs_share = (g["snow_source"] == "observed").mean() if "snow_source" in g else float("nan")
        L.append(f"| {_resort_label(r)} | {len(g)} | {g['winter'].min()} | {g['winter'].max()} | {_fmt(obs_share * 100, 0)} % |")
    L.append("")

    L.append("## Trender (Theil-Sen per decennium, 95 % KI)\n")
    L.append("| Ort | Mått | Period | Medel | Per decennium | KI | Totalt | MK |")
    L.append("|---|---|---|---|---|---|---|---|")
    for _, t in trends.iterrows():
        L.append(
            f"| {_resort_label(t['resort'])} | {METRIC_LABELS.get(t['metric'], t['metric'])} | {t['first_year']}–{t['last_year']} | "
            f"{_fmt(t['mean'])} | {t['sen_slope_per_decade']:+.2f} | [{_fmt(t['sen_lo'], 2)}, {_fmt(t['sen_hi'], 2)}] | "
            f"{t['total_change']:+.1f} | {_sig(t['mk_p'])} |"
        )
    L.append("")

    if not periods.empty:
        p0, p1 = PERIODS["past"], PERIODS["present"]
        L.append(f"## {p0[0]}–{p0[1]} mot {p1[0]}–{p1[1]}\n")
        L.append("| Ort | Mått | Då | Nu | Skillnad | Welch p | Mann-Whitney p |")
        L.append("|---|---|---|---|---|---|---|")
        for _, t in periods.iterrows():
            L.append(
                f"| {_resort_label(t['resort'])} | {METRIC_LABELS.get(t['metric'], t['metric'])} | {_fmt(t['past_mean'])} | "
                f"{_fmt(t['present_mean'])} | {t['diff']:+.1f} | {_fmt(t['welch_p'], 3)} | {_fmt(t['mannwhitney_p'], 3)} |"
            )
        L.append("")

    if not changepoints.empty:
        L.append("## Brytpunkter (nivåskifte, permutationstest)\n")
        L.append("| Ort | Mått | År | Före | Efter | Skifte | p |")
        L.append("|---|---|---|---|---|---|---|")
        for _, t in changepoints.iterrows():
            yr = int(t["year"]) if np.isfinite(t["year"]) else "–"
            L.append(
                f"| {_resort_label(t['resort'])} | {METRIC_LABELS.get(t['metric'], t['metric'])} | {yr} | {_fmt(t['before'])} | "
                f"{_fmt(t['after'])} | {t['shift']:+.1f} | {_fmt(t['p'], 3)} |"
            )
        L.append("")

    if not sensitivity.empty:
        L.append("## Känslighet: säsongsdagar per grad\n")
        L.append("| Ort | Dagar per °C (dec–mar) | SE | p | Dagar per 100 mm nederbörd | R² |")
        L.append("|---|---|---|---|---|---|")
        for _, t in sensitivity.iterrows():
            L.append(
                f"| {_resort_label(t['resort'])} | {t['days_per_degC']:+.1f} | {_fmt(t['days_per_degC_se'])} | {_fmt(t['p_temp'], 3)} | "
                f"{t['days_per_100mm']:+.1f} | {_fmt(t['r2'], 2)} |"
            )
        L.append("")

    if not coupling.empty:
        L.append("## Koppling till global uppvärmning (NASA GISTEMP, DJF)\n")
        L.append("| Ort | Lokal °C per global °C | SE | p | R² | Säsongsdagar per global °C | p |")
        L.append("|---|---|---|---|---|---|---|")
        for _, t in coupling.iterrows():
            L.append(
                f"| {_resort_label(t['resort'])} | {t['local_degC_per_global_degC']:.2f} | {_fmt(t['se'], 2)} | {_fmt(t['p'], 3)} | "
                f"{_fmt(t['r2_temp'], 2)} | {t['season_days_per_global_degC']:+.1f} | {_fmt(t['p_season'], 3)} |"
            )
        L.append("")

    if model_notes:
        L.append("## Projektionsmodell (tidsserie-CV, MAE i dagar)\n")
        L.append("| Ort | Gradient boosting | Ridge | Klimatologi | Vald | Träningsvintrar |")
        L.append("|---|---|---|---|---|---|")
        for m in model_notes:
            L.append(f"| {_resort_label(m.resort)} | {_fmt(m.cv_mae_gbr)} | {_fmt(m.cv_mae_linear)} | {_fmt(m.cv_mae_climatology)} | {m.chosen} | {m.n_train} |")
        L.append("")
        L.append("Om ingen modell slår klimatologin är projektionen för den orten inte värd papperet.\n")

    if not proj_summary.empty:
        L.append("## Projicerad säsongslängd per decennium (dagar ≥30 cm, SSP5-8.5)\n")
        L.append("| Ort | Decennium | Ensemblemedian | Lägsta modell | Högsta modell | Andel vintrar ≥100 dagar |")
        L.append("|---|---|---|---|---|---|")
        for _, t in proj_summary.iterrows():
            L.append(
                f"| {_resort_label(t['resort'])} | {t['decade']}-talet | {_fmt(t['median'], 0)} | {_fmt(t['min_model'], 0)} | "
                f"{_fmt(t['max_model'], 0)} | {_fmt(t['share_reliable'] * 100, 0)} % |"
            )
        L.append("")
        for _, u in unreliable.iterrows():
            d = u["first_unreliable_decade"]
            if d and u.get("already"):
                msg = f"ligger redan på {int(d)}-talet under 50 % vintrar med 100 dagar. Frågan är inte när, utan sedan när – se trendtabellen."
            elif d:
                msg = f"första decenniet där under hälften av modellvintrarna klarar 100-dagarsregeln: **{int(d)}-talet**."
            else:
                msg = "klarar 100-dagarsregeln i majoriteten av modellvintrar hela vägen till 2050."
            L.append(f"- **{_resort_label(u['resort'])}**: {msg}")
        L.append("")
        L.append(
            "Scenariot är SSP5-8.5, det varmaste som finns tillgängligt. Verkligheten ligger sannolikt under. "
            "Bandet mellan modellerna är en spridning, inte ett konfidensintervall.\n"
        )

    if figures:
        L.append("## Figurer\n")
        for f in figures:
            if f is not None:
                L.append(f"![{f.stem}]({f.name})")
        L.append("")

    path = out / "summary.md"
    path.write_text("\n".join(L), encoding="utf-8")
    return path


In [ ]:
%%writefile skiclimate/cli.py
"""Kommandoraden. Tre steg som kan köras var för sig eller i följd.

    python run_pipeline.py download            # SMHI + ERA5 + CMIP6 + GISTEMP
    python run_pipeline.py features            # bygg dygnsserier och vintertabell
    python run_pipeline.py analyze             # statistik, ML, figurer, summary.md
    python run_pipeline.py all --synthetic     # hela kedjan på påhittad data

Allt hamnar under --data-dir (default ./data).
"""

from __future__ import annotations

import argparse
import logging
import sys

import pandas as pd

from . import analysis, features, report, store, synthetic
from .config import RESORTS, Paths

log = logging.getLogger("skiclimate")


def _resorts(arg: str) -> list[str]:
    if arg == "all":
        return list(RESORTS)
    keys = [k.strip() for k in arg.split(",")]
    bad = [k for k in keys if k not in RESORTS]
    if bad:
        sys.exit(f"Okända orter: {bad}. Välj bland {list(RESORTS)}")
    return keys


# ---------------------------------------------------------------------------
# download
# ---------------------------------------------------------------------------


def cmd_download(a: argparse.Namespace, paths: Paths) -> None:
    keys = _resorts(a.resorts)

    if a.synthetic:
        log.warning("SYNTETISK DATA. Ingenting här är uppmätt. Bra för att testa koden, värdelöst för slutsatser.")
        era5, obs, st, cmip = [], [], [], []
        for k in keys:
            e = synthetic.synthetic_era5(k)
            o, s = synthetic.synthetic_smhi(k, e)
            era5.append(e)
            obs.append(o)
            st.append(s)
            if not a.no_cmip6:
                cmip.append(synthetic.synthetic_cmip6(k))
        store.save(pd.concat(era5), paths.raw / "era5_daily")
        store.save(pd.concat(obs), paths.raw / "smhi_obs")
        store.save(pd.concat(st), paths.raw / "smhi_stations")
        if cmip:
            store.save(pd.concat(cmip), paths.raw / "cmip6_daily")
        store.save(synthetic.synthetic_gistemp(), paths.raw / "gistemp")
        return

    from . import openmeteo, smhi  # nätverksmoduler, importeras bara när de behövs

    if not a.no_smhi:
        obs, st = smhi.download_all(keys, hourly=a.smhi_hourly)
        store.save(obs, paths.raw / "smhi_obs")
        store.save(st, paths.raw / "smhi_stations")

    if not a.no_era5:
        frames = []
        for k in keys:
            df, _ = openmeteo.fetch_era5_daily(RESORTS[k])
            frames.append(df)
        store.save(pd.concat(frames, ignore_index=True), paths.raw / "era5_daily")

    if a.era5_snow:
        frames = [openmeteo.fetch_era5_snow_depth(RESORTS[k]) for k in keys]
        store.save(pd.concat(frames, ignore_index=True), paths.raw / "era5_snow")

    if not a.no_cmip6:
        frames = []
        for k in keys:
            df, _ = openmeteo.fetch_cmip6(RESORTS[k])
            frames.append(df)
        store.save(pd.concat(frames, ignore_index=True), paths.raw / "cmip6_daily")

    try:
        store.save(openmeteo.fetch_gistemp(), paths.raw / "gistemp")
    except Exception as e:  # noqa: BLE001
        log.warning("GISTEMP gick inte att hämta (%s). Analysen kör utan global koppling.", e)


# ---------------------------------------------------------------------------
# features
# ---------------------------------------------------------------------------


def cmd_features(a: argparse.Namespace, paths: Paths) -> None:
    keys = _resorts(a.resorts)
    obs = store.load(paths.raw / "smhi_obs")
    st = store.load(paths.raw / "smhi_stations")
    era5 = store.load(paths.raw / "era5_daily")
    era5_snow = store.load(paths.raw / "era5_snow")
    cmip = store.load(paths.raw / "cmip6_daily")
    if obs is None and era5 is None:
        sys.exit("Ingen rådata. Kör `download` först.")

    daily_frames, winter_frames, cmip_frames = [], [], []
    for k in keys:
        target = RESORTS[k].top_elevation if a.elevation == "top" else RESORTS[k].base_elevation
        d = features.build_daily(k, obs, st, era5, era5_snow, target_elevation=target)
        daily_frames.append(d)
        w = features.winter_table(d)
        winter_frames.append(w)
        log.info("%s: %d dygn, %d vintrar (%d–%d)", RESORTS[k].name, len(d), len(w), w["winter"].min(), w["winter"].max())
        if cmip is not None and not cmip.empty:
            cmip_frames.append(features.cmip6_winter_table(cmip, k, target_elevation=target))

    store.save(pd.concat(daily_frames, ignore_index=True), paths.processed / "daily")
    store.save(pd.concat(winter_frames, ignore_index=True), paths.processed / "winters")
    if cmip_frames:
        store.save(pd.concat(cmip_frames, ignore_index=True), paths.processed / "cmip6_winters")


# ---------------------------------------------------------------------------
# analyze
# ---------------------------------------------------------------------------


def cmd_analyze(a: argparse.Namespace, paths: Paths) -> None:
    winters = store.load(paths.processed / "winters")
    if winters is None:
        sys.exit("Ingen vintertabell. Kör `features` först.")
    cmip_w = store.load(paths.processed / "cmip6_winters")
    gistemp = store.load(paths.raw / "gistemp")
    out = paths.output

    trends = analysis.all_trends(winters)
    cps = analysis.all_changepoints(winters)
    periods = analysis.period_comparison(winters)
    sens = analysis.season_sensitivity(winters)
    coupling = analysis.global_coupling(winters, gistemp)
    for name, df in (("trends", trends), ("changepoints", cps), ("period_comparison", periods), ("sensitivity", sens), ("global_coupling", coupling)):
        if not df.empty:
            df.to_csv(out / f"{name}.csv", index=False)

    proj, notes, proj_summary, unreliable = pd.DataFrame(), [], pd.DataFrame(), pd.DataFrame()
    if cmip_w is not None and not cmip_w.empty:
        corrected = analysis.bias_correct(cmip_w, winters)
        proj, notes = analysis.fit_projection(winters, corrected)
        if not proj.empty:
            proj.to_csv(out / "projection_winters.csv", index=False)
            proj_summary = analysis.summarise_projection(proj)
            proj_summary.to_csv(out / "projection_by_decade.csv", index=False)
            unreliable = analysis.first_unreliable_decade(proj_summary)

    figures = [report.plot_metric_trends(winters, m, out) for m in ("season_days_30cm", "t_mean_winter", "thaw_days_core", "snowmaking_days", "thermal_winter_length")]
    figures.append(report.plot_sensitivity(winters, out))
    figures.append(report.plot_global_coupling(winters, gistemp, out))
    figures.append(report.plot_projection(winters, proj, out))

    note = "SYNTETISK DATA – siffrorna nedan är påhittade och finns bara för att visa att pipelinen fungerar." if a.synthetic else ""
    path = report.write_summary(out, winters, trends, cps, periods, sens, coupling, proj_summary, unreliable, notes, [f for f in figures if f], note)
    log.info("Sammanfattning: %s", path)
    print(path.read_text(encoding="utf-8"))


# ---------------------------------------------------------------------------


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="skiclimate", description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("command", choices=("download", "features", "analyze", "all"))
    p.add_argument("--resorts", default="all", help="kommaseparerat: salen,are,tarnaby (default alla)")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--synthetic", action="store_true", help="påhittad data istället för nätverk (test/demo)")
    p.add_argument("--elevation", choices=("base", "top"), default="base", help="vilken höjd säsongen beskrivs på")
    p.add_argument("--no-smhi", action="store_true")
    p.add_argument("--no-era5", action="store_true")
    p.add_argument("--no-cmip6", action="store_true")
    p.add_argument("--era5-snow", action="store_true", help="hämta ERA5-snödjup per timme (långsamt, ~85 anrop per ort)")
    p.add_argument("--smhi-hourly", action="store_true", help="hämta även timdata från SMHI (stora filer)")
    p.add_argument("-v", "--verbose", action="store_true")
    return p


def main(argv: list[str] | None = None) -> None:
    a = build_parser().parse_args(argv)
    logging.basicConfig(level=logging.DEBUG if a.verbose else logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", datefmt="%H:%M:%S")
    paths = Paths(root=__import__("pathlib").Path(a.data_dir)).ensure()
    steps = {"download": cmd_download, "features": cmd_features, "analyze": cmd_analyze}
    for name in (("download", "features", "analyze") if a.command == "all" else (a.command,)):
        log.info("==== %s ====", name)
        steps[name](a, paths)


## 1. Nedladdning

In [ ]:
import sys, logging, importlib
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s', datefmt='%H:%M:%S')

# om du ändrat en modul ovan och kör om: tvinga omladdning
for m in [m for m in list(sys.modules) if m.startswith('skiclimate')]:
    del sys.modules[m]
from skiclimate.cli import main

args = ['--resorts', RESORTS, '--data-dir', 'data']
if SYNTHETIC:
    args.append('--synthetic')
if ERA5_SNOW:
    args.append('--era5-snow')

main(['download', *args])

## 2. Vintertabell

In [ ]:
main(['features', *args, '--elevation', ELEVATION])

## 3. Analys

Skriver `data/output/summary.md`, CSV:er för varje tabell och PNG-figurer.

In [ ]:
main(['analyze', *args])

## Resultat

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

winters = pd.read_parquet('data/processed/winters.parquet')
display(winters.groupby('resort')[['season_days_30cm', 't_mean_winter', 'thaw_days_core', 'snowmaking_days']].describe().T.round(1))

for f in ['trend_season_days_30cm', 'trend_t_mean_winter', 'trend_thaw_days_core', 'sensitivity', 'global_coupling', 'projection_season_days_30cm']:
    p = f'data/output/{f}.png'
    if os.path.exists(p):
        display(Image(p))

In [ ]:
display(Markdown(open('data/output/summary.md', encoding='utf-8').read()))

## Egen analys

`winters` har en rad per ort och vinter. `data/processed/daily.parquet` har dygnsvärdena. `data/output/projection_winters.csv` har varje CMIP6-modells projicerade vintrar. Alla funktioner i `skiclimate.analysis` tar vanliga DataFrames.

In [ ]:
from skiclimate import analysis

analysis.season_sensitivity(winters)   # säsongsdagar per grad, per ort